# Notebook 04 — Phase 3: combined-machine model (GPU, resume-safe)
**Needs:** notebook 02 (`prepared_v3`) and notebook 03 (`revision/machine_v3`, Experiments 1 and 2) already finished.

The study design, Phase 3: *train using data from Machine A and Machine B, then evaluate on patient-level held-out data from
both domains. This tests whether exposure to multiple machines improves generalization.*

**Design (fixed before running), aligned exactly with Experiments 1 and 2:**
* Same 344 patients, same 5 within-machine folds (notebook 02), same tasks, backbones, initialisations, seeds, inner 15 %
  early-stopping split, augmentation and hyper-parameters as notebook 03.
* Fold k: the model is trained on folds ≠ k of **both** machines and tested on fold k of Machine 1 **and** fold k of Machine 2.
  These are exactly the patients that the within-machine models of Experiment 1 were tested on, so the comparison is paired.
* Two versions of the combined model:
  * **full** — all training-fold patients of both machines (about twice the single-machine training set).
  * **match** — the same number of patients as the *smaller* single-machine training set, half from each machine
    (class-stratified random sample). It never has more training patients than either single-machine model, so a gain
    cannot come from more data — only from seeing both machines.
* 2 versions × 5 tasks × 4 backbones × 2 initialisations × 5 folds × 3 seeds = 1,200 runs. Seed 42 first.

**Analysis (last cells, CPU, works on partial results):** for every test machine A — within (Exp 1, trained on A only),
cross (Exp 2, trained on the other machine only), combined-full and combined-match; contrasts combined − within and
combined − cross with a joint patient bootstrap (10,000 resamples), Holm correction, pooled results per initialisation.

**Runtime:** GPU → Run all. First `QUICK_TEST = True` (≈5 min), then `False`. Safe to re-run: finished runs are skipped.

In [ ]:
# settings
import os, sys, subprocess, time, copy, random, json, gc
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'huggingface_hub', 'python-docx'], check=False)
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
import timm
from PIL import Image
from sklearn.model_selection import train_test_split

LOCAL_TEST = bool(os.environ.get('BONESCAN_BASE'))
if LOCAL_TEST:
    BASE = os.environ['BONESCAN_BASE']
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Colab Notebooks/bonescan'
PREP, CV_IN, BSD = BASE + '/prepared', BASE + '/cv5_results', BASE + '/bs80k'
EXP12 = BASE + '/revision/machine_v3'            # Experiments 1 + 2 (notebook 03) - used for the comparison
OUT = BASE + '/revision/phase3'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'NOT FOUND - select Runtime > Change runtime type > GPU')

QUICK_TEST = True            # first True (short check), then False
MAX_HOURS = 11.5             # stop after this many hours; run all again to resume (finished runs are skipped)
RUN_TRAINING = True          # set False to re-run only the analysis

MODELS = ['tf_efficientnetv2_s', 'resnext50_32x4d', 'regnety_016', 'inception_v3']
TASKS = {
    '3class':              {'SECONDARY': 0, 'DEGENERATIVE': 1, 'NORMAL': 2},
    'cancer_vs_noncancer': {'SECONDARY': 1, 'DEGENERATIVE': 0, 'NORMAL': 0},
    'sec_vs_normal':       {'SECONDARY': 1, 'NORMAL': 0},
    'sec_vs_degen':        {'SECONDARY': 1, 'DEGENERATIVE': 0},
    'degen_vs_normal':     {'DEGENERATIVE': 1, 'NORMAL': 0},
}
TASK_ID = {'3class': 'T1', 'cancer_vs_noncancer': 'T2', 'sec_vs_normal': 'T3', 'sec_vs_degen': 'T4', 'degen_vs_normal': 'T5'}
MN = {'tf_efficientnetv2_s': 'EfficientNetV2-S', 'resnext50_32x4d': 'ResNeXt50', 'regnety_016': 'RegNetY-1.6GF', 'inception_v3': 'Inception-v3'}
SEEDS = [42, 43, 44]
INITS = ['IN', 'BS']                      # IN = ImageNet, BS = BS-80K
CAMERAS = ['M1', 'M2']
VARIANTS = ['full', 'match']
N_BOOT = 10000

N_FOLDS, EPOCHS, PATIENCE, BATCH, H, W = 5, 30, 8, 16, 512, 176
INNER_FRAC = 0.15
RAD_REPO, RAD_FILE = 'Lab-Rasool/RadImageNet', 'InceptionV3.pt'     # not used here (shared code)
PRETRAINED = not LOCAL_TEST

if QUICK_TEST:
    MODELS, TASKS = ['regnety_016'], {'sec_vs_normal': TASKS['sec_vs_normal'], '3class': TASKS['3class']}
    SEEDS, EPOCHS, N_BOOT = [42], 1, 200
    OUT = BASE + '/revision/phase3_quick'
QUICK_FOLDS = list(range(N_FOLDS))
for d in ['preds', 'logs', 'analysis']:
    os.makedirs(f'{OUT}/{d}', exist_ok=True)
RUNS = OUT + '/runs.csv'
T_START = time.time()
assert os.path.isdir(EXP12 + '/preds'), 'results of notebook 03 (revision/machine_v3) not found'
print('OUT:', OUT, '| Exp 1/2 from:', EXP12, '| QUICK_TEST:', QUICK_TEST)

In [ ]:
# data (prepared_v3 = the final dataset, made by notebook 02)
P3 = BASE + '/prepared_v3'
fix = lambda p: P3 + '/images/' + os.path.basename(str(p))
allp = pd.read_csv(P3 + '/patients.csv')
sp = allp[allp.split == 'pool'][['patient_id', 'machine', 'label', 'ANT', 'POST', 'fold']].reset_index(drop=True)
sp['fold'] = sp.fold.astype(int); sp['split'] = 'pool'      # no separate test set: 5-fold CV inside each machine
assert sp.patient_id.is_unique and sp.label.isin(['SECONDARY', 'DEGENERATIVE', 'NORMAL']).all()
print('classification patients:', len(sp), '| Ind not used:', (allp.split == 'ind').sum(), '| excluded by audit:', (allp.split == 'excluded').sum())
print(pd.crosstab(sp.label, sp.machine, margins=True))

ALL = sp[['patient_id', 'ANT', 'POST', 'machine']].reset_index(drop=True)
IDX = {p: i for i, p in enumerate(ALL.patient_id)}
CAM = dict(zip(ALL.patient_id, ALL.machine))
def load_img(p):
    return np.array(Image.open(fix(p)).convert('L'))
X = torch.from_numpy(np.stack([np.stack([load_img(a), load_img(b)]) for a, b in zip(ALL.ANT, ALL.POST)]))   # N x 2 x H x W uint8
assert X.shape[2:] == (H, W), X.shape
X = X.to(DEVICE)
print('images on', DEVICE, tuple(X.shape), f'{X.numel() / 1e6:.0f} MB')

In [ ]:
# model, GPU augmentation, training
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def augment(img, rng):
    """Same augmentation as the paper (RandomAffine ±7°, ±4 %, 0.93–1.07, white fill; brightness/contrast ±20 %), done on the GPU per image."""
    out = []
    for x in img:                                                   # x: 1 x H x W in [0,1]
        ang = float(rng.uniform(-7, 7)); sc = float(rng.uniform(0.93, 1.07))
        tx, ty = int(round(rng.uniform(-0.04, 0.04) * W)), int(round(rng.uniform(-0.04, 0.04) * H))
        x = TF.affine(x, angle=ang, translate=[tx, ty], scale=sc, shear=[0.0], interpolation=InterpolationMode.NEAREST, fill=[1.0])
        ops = [('b', float(rng.uniform(0.8, 1.2))), ('c', float(rng.uniform(0.8, 1.2)))]
        if rng.random() < 0.5:
            ops = ops[::-1]
        for k, f in ops:
            x = (x * f).clamp(0, 1) if k == 'b' else ((x - x.mean()) * f + x.mean()).clamp(0, 1)
        out.append(x)
    return torch.stack(out)

def to_input(idx, train, rng=None):
    x = X[idx].float() / 255.0                                       # B x 2 x H x W
    a, p = x[:, :1], x[:, 1:]
    if train:
        a, p = augment(a, rng), augment(p, rng)
    norm = lambda t: (t.repeat(1, 3, 1, 1) - MEAN) / STD
    return norm(a), norm(p)

class Pooled(nn.Module):
    """Wraps any feature extractor (e.g. RadImageNet full-model object) so that it returns B x C."""
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        y = self.m(x)
        if isinstance(y, (tuple, list)):
            y = y[0]
        if hasattr(y, 'logits'):
            y = y.logits
        return F.adaptive_avg_pool2d(y, 1).flatten(1) if y.dim() == 4 else y.flatten(1)

_RAD_CACHE = {}
def radimagenet_backbone():
    if 'm' not in _RAD_CACHE:
        if LOCAL_TEST:
            path = BASE + '/radimagenet_test.pt'
        else:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download(repo_id=RAD_REPO, filename=RAD_FILE)
        obj = torch.load(path, map_location='cpu', weights_only=False)
        if isinstance(obj, dict):                                       # file is a state_dict -> rebuild Inception-v3 and load it
            sd = obj.get('state_dict', obj.get('model', obj))
            for pre in ['module.', 'backbone.', 'model.', 'base_model.', 'features.']:
                if all(k.startswith(pre) for k in sd):
                    sd = {k[len(pre):]: v for k, v in sd.items()}
            print('RadImageNet file is a state_dict with', len(sd), 'tensors, e.g.', list(sd)[:3])
            best = None
            for builder in ['timm', 'torchvision']:
                cand = timm.create_model('inception_v3', pretrained=False, num_classes=0) if builder == 'timm' else __import__('torchvision').models.inception_v3(weights=None, aux_logits=False, init_weights=False)
                msd = cand.state_dict()
                ok = {k: v for k, v in sd.items() if k in msd and msd[k].shape == v.shape}
                isbb = lambda k: not k.startswith(('fc.', 'AuxLogits'))
                frac = sum(v.numel() for k, v in ok.items() if isbb(k)) / sum(v.numel() for k, v in msd.items() if isbb(k))
                print(f'  {builder} inception_v3: {len(ok)}/{len(msd)} tensors match ({frac:.0%} of backbone weights)')
                if best is None or frac > best[0]:
                    best = (frac, builder, cand, ok)
            frac, builder, m, ok = best
            if frac < 0.95:                                             # saved as nn.Sequential ('0.conv.weight', ...): map by position + shape
                cand = timm.create_model('inception_v3', pretrained=False, num_classes=0); msd = cand.state_dict()
                tk = [k for k in msd if not k.startswith(('fc.', 'AuxLogits'))]
                sv = list(sd.values())
                if len(tk) == len(sv) and all(msd[a].shape == v.shape for a, v in zip(tk, sv)):
                    m, ok, frac, builder = cand, dict(zip(tk, sv)), 1.0, 'timm (by position)'
                    print(f'  positional mapping: {len(tk)}/{len(tk)} tensors, all shapes match')
            assert frac > 0.95, ('RadImageNet weights do not match Inception-v3 (only %.0f%% loaded) - check the RadImageNet weight file' % (100 * frac))
            m.load_state_dict(ok, strict=False)
            print(f'  loaded into {builder} inception_v3 ({frac:.0%} of weights)')
        elif isinstance(obj, torch.jit.ScriptModule):
            m = obj
        else:
            m = obj
        for attr in ['fc', 'classifier', 'head']:                       # drop a classification head if present
            if hasattr(m, attr) and isinstance(getattr(m, attr), nn.Module):
                setattr(m, attr, nn.Identity())
        if hasattr(m, 'aux_logits'):
            m.aux_logits = False; m.AuxLogits = None
        _RAD_CACHE['m'] = m
        print('RadImageNet backbone:', type(m).__name__, '| file:', path)
    return Pooled(copy.deepcopy(_RAD_CACHE['m']))

def make_backbone(name, init):
    if init == 'RAD':
        return radimagenet_backbone()
    bb = timm.create_model(name, pretrained=PRETRAINED and init != 'SCRATCH', num_classes=0)
    if init == 'IN':
        return bb
    f = (f'{BSD}/model_bs80k_{name}.pt' if init == 'BS' else
         f'{OUT}/models/model_bs80k_{name}_frac{init[3:]}.pt' if init.startswith('BSf') else       # 'BSf0.10' (independent subsets)
         f'{OUT}/models/model_bs80k_{name}_nested{init[3:]}.pt')                                     # 'BSn0.10' (nested subsets, equal steps)
    sd = torch.load(f, map_location='cpu')
    res = bb.load_state_dict({k[len('backbone.'):]: v for k, v in sd.items() if k.startswith('backbone.')}, strict=False)
    assert not res.missing_keys, f'missing keys in {f}: {res.missing_keys[:3]}'
    return bb

class DualViewNet(nn.Module):
    def __init__(self, name, n, init):
        super().__init__()
        self.backbone = make_backbone(name, init)
        self.backbone.eval()
        with torch.no_grad():
            c = self.backbone(torch.zeros(1, 3, H, W)).shape[1]
        self.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(2 * c, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n))
    def features(self, a, p):
        return torch.cat([self.backbone(a), self.backbone(p)], 1)
    def forward(self, a, p):
        return self.head(self.features(a, p))

def predict(model, idx, amp, feats=False):
    model.eval(); P, Fz = [], []
    with torch.no_grad(), torch.autocast(device_type='cuda', enabled=amp):
        for i in range(0, len(idx), 32):
            a, p = to_input(idx[i:i + 32], False)
            z = model.features(a, p); P.append(F.softmax(model.head(z).float(), 1).cpu())
            if feats:
                Fz.append(z.float().cpu())
    P = torch.cat(P).numpy() if P else np.zeros((0, 1))
    return (P, torch.cat(Fz).numpy().astype(np.float32)) if feats else P

def macro_f1(pred, y, n):
    f = []
    for c in range(n):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))

def train(name, init, n, tr_idx, tr_y, va_idx, va_y, seed, epochs=EPOCHS, patience=PATIENCE, amp=True):
    set_seed(seed); amp = amp and DEVICE == 'cuda'
    model = DualViewNet(name, n, init).to(DEVICE)
    cnt = np.bincount(tr_y, minlength=n)
    w = torch.tensor(len(tr_y) / (n * np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    opt = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': 1e-4}, {'params': model.head.parameters(), 'lr': 1e-3}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=amp)
    rng = np.random.default_rng(seed); yt = torch.tensor(tr_y, device=DEVICE)
    best, state, wait, best_ep, ep = -1.0, None, 0, 0, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = rng.permutation(len(tr_idx)); nb = max(len(perm) // BATCH, 1)
        for b in range(nb):
            j = perm[b * BATCH:(b + 1) * BATCH]
            a, p = to_input(tr_idx[j], True, rng)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=amp):
                loss = loss_fn(model(a, p), yt[j])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        pv = predict(model, va_idx, amp)
        vf = macro_f1(pv.argmax(1), va_y, n) if not np.isnan(pv).any() else -1
        if vf > best:
            best, state, wait, best_ep = vf, copy.deepcopy(model.state_dict()), 0, ep
        else:
            wait += 1
            if wait >= patience:
                break
    if state is not None:
        model.load_state_dict(state)
    return model, dict(best_inner_f1=best, best_epoch=best_ep, epochs_run=ep, amp=amp)

def ids_to_idx(ids):
    return np.array([IDX[p] for p in ids])

def save_preds(path, P, ids, y=None):
    d = pd.DataFrame(P, columns=[f'prob_{i}' for i in range(P.shape[1])])
    if y is not None:
        d.insert(0, 'y_true', y)
    d.insert(0, 'patient_id', list(ids)); d.insert(1, 'camera', [CAM[p] for p in ids]); d.to_csv(path, index=False)

def done_keys():
    return {f[:-5] for f in os.listdir(f'{OUT}/logs') if f.endswith('.json')}

def log_run(row):
    # one small JSON per finished run (safe if Colab disconnects); runs.csv is rebuilt from all of them
    with open(f"{OUT}/logs/{row['key']}.json", 'w') as fh:
        json.dump({k: (v.item() if hasattr(v, 'item') else v) for k, v in row.items()}, fh)
    rows = [json.load(open(f'{OUT}/logs/{f}')) for f in sorted(os.listdir(f'{OUT}/logs')) if f.endswith('.json')]
    pd.DataFrame(rows).to_csv(RUNS, index=False)

def time_left():
    return MAX_HOURS - (time.time() - T_START) / 3600

def run_job(key, exp, name, init, n, seed, tr, va, evals, feat_sets=(), extra=None):
    """tr/va: DataFrames with patient_id and y. evals: {kind: (ids, y or None)}. Saves preds of every eval set."""
    t0 = time.time()
    for amp in [True, False]:
        model, info = train(name, init, n, ids_to_idx(tr.patient_id), tr.y.values, ids_to_idx(va.patient_id), va.y.values, seed, amp=amp)
        outs, bad = {}, False
        for kind, (ids, y) in evals.items():
            res = predict(model, ids_to_idx(ids), info['amp'], feats=kind in feat_sets)
            P = res[0] if kind in feat_sets else res
            bad |= bool(np.isnan(P).any()); outs[kind] = res
        if not bad or not amp or DEVICE != 'cuda':
            break
        print('  NaN -> retrying in full precision'); del model; gc.collect(); torch.cuda.empty_cache()
    for kind, (ids, y) in evals.items():
        res = outs[kind]; P = res[0] if kind in feat_sets else res
        save_preds(f'{OUT}/preds/{key}_{kind}.csv', P, ids, y)
        if kind in feat_sets:
            np.savez_compressed(f'{OUT}/feats/{key}_{kind}.npz', ids=np.array(ids), z=np.clip(res[1], -6e4, 6e4).astype(np.float16))
    row = dict(key=key, exp=exp, model=name, init=init, seed=seed, minutes=round((time.time() - t0) / 60, 2), n_train=len(tr), n_inner=len(va), **info, **(extra or {}))
    for kind, (ids, y) in evals.items():
        if y is not None:
            res = outs[kind]; P = res[0] if kind in feat_sets else res
            row[f'{kind}_f1'] = macro_f1(P.argmax(1), np.asarray(y), n)
    log_run(row)
    print(f"  {key}: inner {info['best_inner_f1']:.3f} | " + ' | '.join(f"{k[:-3]} {v:.3f}" for k, v in row.items() if k.endswith('_f1') and k != 'best_inner_f1') + f" | {row['minutes']} min", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()

def inner_split(df, seed=0):
    for strat in [df.label + '_' + df.machine, df.label, None]:     # class x camera; fall back if a stratum is too small
        try:
            tr, va = train_test_split(df, test_size=INNER_FRAC, stratify=strat, random_state=seed)
            return tr.reset_index(drop=True), va.reset_index(drop=True)
        except ValueError:
            continue

def eta(exp, remaining):
    if not os.path.exists(RUNS):
        return ''
    r = pd.read_csv(RUNS); r = r[r.exp == exp] if 'exp' in r else r.iloc[:0]
    return f'(~{remaining * r.minutes.mean() / 60:.1f} h left for {exp})' if len(r) else ''
print('Ready')

In [ ]:
# Phase 3: train on both machines (folds != k), test on fold k of Machine 1 and fold k of Machine 2
def inner_seed(k, seed):
    return 1000 + k + 100 * (seed - 42)          # identical in all notebooks

def match_seed(k, seed):
    return 3000 + k + 100 * (seed - 42)

PATIENTS = sp.copy()
FOLD = dict(zip(PATIENTS.patient_id, PATIENTS.fold))

def train_pool(d, k, variant, seed):
    """d: patients of one task. full = all training-fold patients of both machines;
    match = as many patients as the smaller single-machine training set, half from each machine, class-stratified."""
    pool = d[d.fold != k]
    if variant == 'full':
        return pool
    n_tot = int(pool.machine.value_counts().reindex(CAMERAS, fill_value=0).min())
    parts = []
    for c, n_c in [('M1', n_tot - n_tot // 2), ('M2', n_tot // 2)]:
        pc = pool[pool.machine == c]
        if n_c >= len(pc):
            parts.append(pc); continue
        try:
            sub, _ = train_test_split(pc, train_size=n_c, stratify=pc.label, random_state=match_seed(k, seed))
        except ValueError:
            sub, _ = train_test_split(pc, train_size=n_c, random_state=match_seed(k, seed))
        parts.append(sub)
    out = pd.concat(parts)
    assert len(out) == n_tot
    return out

def p3_jobs():
    jobs = []
    for seed in SEEDS:                           # seed 42 for everything first
        for variant in VARIANTS:
            for init in INITS:
                for task in TASKS:
                    for name in MODELS:
                        for k in QUICK_FOLDS:
                            jobs.append((f'p3_{variant}_{task}_{name}_{init}_s{seed}_fold{k}', variant, task, name, init, seed, k))
    return jobs

def run_p3():
    jobs = p3_jobs(); done = done_keys(); todo = [j for j in jobs if j[0] not in done]
    print(f'phase3: {len(jobs) - len(todo)}/{len(jobs)} done', eta('phase3', len(todo)))
    for key, variant, task, name, init, seed, k in todo:
        if time_left() <= 0:
            print('MAX_HOURS reached - restart the runtime and run all again to resume'); return False
        mp = TASKS[task]; n = len(set(mp.values()))
        d = PATIENTS[PATIENTS.label.isin(mp)].assign(y=lambda x: x.label.map(mp))
        pool = train_pool(d, k, variant, seed)
        tr, inner = inner_split(pool, seed=inner_seed(k, seed))
        te = {c: d[(d.machine == c) & (d.fold == k)] for c in CAMERAS}
        test_ids = set(te['M1'].patient_id) | set(te['M2'].patient_id)
        assert not (set(tr.patient_id) | set(inner.patient_id)) & test_ids
        assert set(tr.patient_id).isdisjoint(inner.patient_id) and (pool.fold != k).all()
        evals = {'tM1': (te['M1'].patient_id, te['M1'].y.values), 'tM2': (te['M2'].patient_id, te['M2'].y.values),
                 'inner': (inner.patient_id, inner.y.values)}
        nm = pool.machine.value_counts().to_dict()
        print(f'[phase3-{variant}] {task} | {name} | {init} | seed {seed} | fold {k}  '
              f'(train {len(tr)} + inner {len(inner)}: M1 {nm.get("M1", 0)}, M2 {nm.get("M2", 0)} | test M1 {len(te["M1"])}, M2 {len(te["M2"])})')
        run_job(key, 'phase3', name, init, n, seed, tr, inner, evals,
                extra=dict(task=task, variant=variant, fold=k, n_pool_M1=nm.get('M1', 0), n_pool_M2=nm.get('M2', 0), inner_seed=inner_seed(k, seed)))
    return True

OK = run_p3() if RUN_TRAINING else True
if os.path.exists(RUNS):
    r = pd.read_csv(RUNS)
    print(r.groupby(['exp', 'variant']).agg(runs=('key', 'count'), minutes=('minutes', 'sum')).assign(hours=lambda d: (d.minutes / 60).round(1)))
print('ALL RUNS FINISHED' if OK else 'NOT FINISHED YET - Run all again (finished runs are skipped); the analysis below uses what is done')

In [ ]:
# Analysis 1: collect predictions (every patient is predicted once per seed by the fold-k model of its own fold)
def rd(path):
    return pd.read_csv(path) if os.path.exists(path) else None

def pcols(df):
    return sorted([c for c in df.columns if c.startswith('prob_')], key=lambda s: int(s[5:]))

def other(c):
    return 'M2' if c == 'M1' else 'M1'

def f_p3(variant, task, name, init, s, k, cam):
    return f'{OUT}/preds/p3_{variant}_{task}_{name}_{init}_s{s}_fold{k}_t{cam}.csv'

def f_e12(task, name, init, train_cam, s, k, kind):
    return f'{EXP12}/preds/mach_{task}_{name}_{init}_{train_cam}_s{s}_fold{k}_{kind}.csv'

def seed_ok(variant, task, name, init, s):
    return all(os.path.exists(f_p3(variant, task, name, init, s, k, c)) and os.path.exists(f_e12(task, name, init, c, s, k, 'oof'))
               and os.path.exists(f_e12(task, name, init, c, s, k, 'cross')) for k in QUICK_FOLDS for c in CAMERAS)

def load_arm(arm, variant, task, name, init, s, cam):
    """predictions for the patients of machine `cam`, one per patient, from one seed"""
    L = []
    for k in QUICK_FOLDS:
        if arm == 'comb':
            x = rd(f_p3(variant, task, name, init, s, k, cam))
        elif arm == 'within':
            x = rd(f_e12(task, name, init, cam, s, k, 'oof'))
        else:                                                   # cross: model trained on the other machine, fold-k model for fold-k patients
            x = rd(f_e12(task, name, init, other(cam), s, k, 'cross'))
            x = x[x.patient_id.map(FOLD) == k]
        assert (x.patient_id.map(FOLD) == k).all() and (x.camera == cam).all(), (arm, task, name, init, s, k, cam)
        L.append(x)
    return pd.concat(L).sort_values('patient_id').reset_index(drop=True)

def avg(lst):
    out = lst[0][['patient_id', 'camera', 'y_true']].copy(); pc = pcols(lst[0])
    for x in lst:
        assert (x.patient_id.values == out.patient_id.values).all() and (x.y_true.values == out.y_true.values).all()
    out[pc] = np.mean([x[pc].values for x in lst], 0); return out

DATA = {}
GROUPS = [(v, t, m, i) for v in VARIANTS for t in TASKS for m in MODELS for i in INITS]
for v, t, m, i in GROUPS:
    seeds = [s for s in SEEDS if seed_ok(v, t, m, i, s)]
    if not seeds:
        continue
    for c in CAMERAS:
        arms = {a: avg([load_arm(a, v, t, m, i, s, c) for s in seeds]) for a in ['within', 'cross', 'comb']}
        assert (arms['within'].patient_id.values == arms['comb'].patient_id.values).all()
        assert (arms['cross'].patient_id.values == arms['comb'].patient_id.values).all()
        DATA[(v, t, m, i, c)] = dict(arms, seeds=seeds)
print(f'complete groups: {len(DATA) // 2}/{len(GROUPS)}  (each tested on both machines)')

In [ ]:
# Analysis 2: metrics + joint patient bootstrap (same resampling scheme as notebook 03)
from sklearn.metrics import roc_auc_score
cam_patients = {c: sorted(PATIENTS[PATIENTS.machine == c].patient_id) for c in CAMERAS}
cam_pos = {c: {p: i for i, p in enumerate(cam_patients[c])} for c in CAMERAS}
rng = np.random.default_rng(2026)
BOOT = {c: np.stack([np.bincount(rng.integers(0, len(cam_patients[c]), len(cam_patients[c])), minlength=len(cam_patients[c]))
                     for _ in range(N_BOOT)]).astype(np.float32) for c in CAMERAS}

def weighted_f1(df, cam, wts):
    pc = pcols(df); y = df.y_true.values.astype(int); pr = df[pc].values.argmax(1); n = len(pc)
    Wm = wts[:, [cam_pos[cam][p] for p in df.patient_id]]
    f1 = []
    for j in range(n):
        tp = Wm @ ((pr == j) & (y == j)).astype(np.float32); fp = Wm @ ((pr == j) & (y != j)).astype(np.float32)
        fn = Wm @ ((pr != j) & (y == j)).astype(np.float32)
        f1.append(np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), 0.0))
    return np.mean(f1, 0)

def point(df):
    pc = pcols(df); y = df.y_true.values.astype(int); P = df[pc].values; pr = P.argmax(1); n = len(pc)
    f1 = macro_f1(pr, y, n); rec = [np.mean(pr[y == j] == j) if (y == j).any() else np.nan for j in range(n)]
    try:
        auc = roc_auc_score(y, P[:, 1]) if n == 2 else roc_auc_score(y, P, multi_class='ovr', average='macro')
    except ValueError:
        auc = np.nan
    out = dict(n=len(y), f1=f1, bacc=float(np.nanmean(rec)), auc=auc)
    if n == 2:
        out.update(sens=rec[1], spec=rec[0])
    return out

def pval(d):
    return min(1.0, 2 * min((np.sum(d <= 0) + 1) / (len(d) + 1), (np.sum(d >= 0) + 1) / (len(d) + 1)))

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for r_, i in enumerate(o):
        run = max(run, (m - r_) * p[i]); adj[i] = min(1.0, run)
    return adj

ROWS, DIST = [], {}
for (v, task, name, init, cam), r_ in DATA.items():
    pt = {a: point(r_[a]) for a in ['within', 'cross', 'comb']}
    bs = {a: weighted_f1(r_[a], cam, BOOT[cam]) for a in ['within', 'cross', 'comb']}
    DIST[(v, task, name, init, cam)] = bs
    row = dict(variant=v, task=TASK_ID[task], task_key=task, model=MN[name], init=init, test_machine=cam, seeds=len(r_['seeds']), n=pt['comb']['n'])
    for a in ['within', 'cross', 'comb']:
        row[f'{a}_f1'] = pt[a]['f1']; row[f'{a}_lo'] = np.percentile(bs[a], 2.5); row[f'{a}_hi'] = np.percentile(bs[a], 97.5)
    for a, b in [('comb', 'within'), ('comb', 'cross')]:
        d = bs[a] - bs[b]
        row[f'{a}_minus_{b}'] = pt[a]['f1'] - pt[b]['f1']; row[f'{a}_minus_{b}_lo'] = np.percentile(d, 2.5)
        row[f'{a}_minus_{b}_hi'] = np.percentile(d, 97.5); row[f'{a}_minus_{b}_p'] = pval(d)
    for a in ['within', 'cross', 'comb']:
        for m_ in ['bacc', 'auc', 'sens', 'spec']:
            if m_ in pt[a]:
                row[f'{a}_{m_}'] = pt[a][m_]
    ROWS.append(row)
A = pd.DataFrame(ROWS)
if len(A):
    for c_ in ['comb_minus_within', 'comb_minus_cross']:
        A[f'{c_}_pholm'] = np.nan
        for (v, ini), g in A.groupby(['variant', 'init']):      # family: 20 task x backbone pairs x 2 test machines
            A.loc[g.index, f'{c_}_pholm'] = holm(g[f'{c_}_p'].values)
    A = A.sort_values(['variant', 'init', 'task', 'model', 'test_machine']).reset_index(drop=True)
    A.to_csv(f'{OUT}/analysis/P3_per_pair.csv', index=False)
    show = ['variant', 'task', 'model', 'init', 'test_machine', 'seeds', 'within_f1', 'cross_f1', 'comb_f1', 'comb_minus_within', 'comb_minus_within_pholm', 'comb_minus_cross']
    print(A[show].round(3).to_string(index=False))

In [ ]:
# Analysis 3: pooled results per version, initialisation and test machine
P_ROWS = []
MLAB = {'M1': 'Machine 1', 'M2': 'Machine 2', 'both': 'both machines'}
def f1_of(key, arm):
    v, t, m, i, c = key
    return A[(A.variant == v) & (A.task_key == t) & (A.model == MN[m]) & (A.init == i) & (A.test_machine == c)][f'{arm}_f1'].iloc[0]

for v in VARIANTS:
    for ini in INITS:
        for mc in ['M1', 'M2', 'both']:
            keys = [k for k in DIST if k[0] == v and k[3] == ini and (mc == 'both' or k[4] == mc)]
            if not keys:
                continue
            mean = {a: np.mean([f1_of(k, a) for k in keys]) for a in ['within', 'cross', 'comb']}
            bd = {a: np.mean([DIST[k][a] for k in keys], 0) for a in ['within', 'cross', 'comb']}
            sub = A[(A.variant == v) & (A.init == ini) & ((A.test_machine == mc) if mc != 'both' else A.test_machine.notna())]
            row = dict(version=v, init=ini, test_machine=MLAB[mc], n_pairs=len(keys),
                       within_f1=mean['within'], cross_f1=mean['cross'], comb_f1=mean['comb'])
            for a, b in [('comb', 'within'), ('comb', 'cross')]:
                d = bd[a] - bd[b]
                row[f'{a}_minus_{b}'] = mean[a] - mean[b]; row[f'{a}_minus_{b}_lo'] = np.percentile(d, 2.5)
                row[f'{a}_minus_{b}_hi'] = np.percentile(d, 97.5); row[f'{a}_minus_{b}_p'] = pval(d)
                row[f'{a}_minus_{b}_n_positive'] = int((sub[f'{a}_minus_{b}'] > 0).sum())
                row[f'{a}_minus_{b}_n_sig_holm'] = int((sub[f'{a}_minus_{b}_pholm'] < 0.05).sum())
            P_ROWS.append(row)
PO = pd.DataFrame(P_ROWS)
if len(PO):
    PO.to_csv(f'{OUT}/analysis/P3_pooled.csv', index=False)
    print(PO[['version', 'init', 'test_machine', 'n_pairs', 'within_f1', 'cross_f1', 'comb_f1', 'comb_minus_within', 'comb_minus_within_lo',
              'comb_minus_within_hi', 'comb_minus_within_p', 'comb_minus_cross', 'comb_minus_cross_p']].round(3).to_string(index=False))
    print('\nwithin = trained on the test machine only (Exp 1); cross = trained on the other machine only (Exp 2); comb = trained on both (Phase 3).')
    print('Positive comb − within = combined model better than the single-machine model on the same patients.', N_BOOT, 'bootstrap resamples.')

In [ ]:
# Analysis 4: Word report + short text summary
from docx import Document
from docx.shared import Pt
def tbl(doc, df, cols, heads, fmt=3, size=7.5):
    t = doc.add_table(rows=1, cols=len(cols)); t.style = 'Table Grid'
    for i, h in enumerate(heads):
        c = t.rows[0].cells[i]; c.text = ''; r = c.paragraphs[0].add_run(h); r.bold = True; r.font.size = Pt(size)
    for _, row in df.iterrows():
        cs = t.add_row().cells
        for i, c in enumerate(cols):
            v = row[c] if not callable(c) else c(row)
            s = (f'{v:.{fmt}f}' if isinstance(v, (float, np.floating)) and not np.isnan(v) else ('–' if isinstance(v, (float, np.floating)) else str(v)))
            cs[i].text = ''; cs[i].paragraphs[0].add_run(s).font.size = Pt(size)
    doc.add_paragraph()
if len(A):
    dci = lambda a: (lambda r: f"{r[a]:+.3f} [{r[a + '_lo']:+.3f}, {r[a + '_hi']:+.3f}]")
    doc = Document(); doc.styles['Normal'].font.name = 'Calibri'; doc.styles['Normal'].font.size = Pt(10)
    doc.add_heading('Phase 3: combined-machine model vs single-machine models (notebook 04)', 0)
    doc.add_paragraph(f'Groups complete: {len(DATA) // 2}/{len(GROUPS)}. Every patient is tested once per seed by the fold model of its own fold; '
                      'predictions are averaged over seeds. within = model trained on the test machine only (Experiment 1); cross = model trained on '
                      'the other machine only (Experiment 2); comb = model trained on both machines (full = all training-fold patients; match = '
                      'as many patients as the smaller single-machine training set, half from each machine). All arms are evaluated on the same '
                      f'patients. 95 % CI and p from a joint patient bootstrap ({N_BOOT} resamples); Holm correction over the 40 pairs of each '
                      'version and initialisation.')
    doc.add_heading('1. Pooled results (macro-F1)', 1)
    tbl(doc, PO, ['version', 'init', 'test_machine', 'n_pairs', 'within_f1', 'cross_f1', 'comb_f1', dci('comb_minus_within'), 'comb_minus_within_p',
                  dci('comb_minus_cross'), 'comb_minus_cross_p'],
        ['Version', 'Init', 'Test machine', 'Pairs', 'Within', 'Cross', 'Combined', 'Comb − within [CI]', 'p', 'Comb − cross [CI]', 'p'], size=7)
    doc.add_heading('2. Every task × backbone × test machine', 1)
    tbl(doc, A, ['variant', 'task', 'model', 'init', 'test_machine', 'seeds', 'within_f1', 'cross_f1', 'comb_f1', dci('comb_minus_within'),
                 'comb_minus_within_pholm', dci('comb_minus_cross'), 'comb_minus_cross_pholm'],
        ['Version', 'Task', 'Backbone', 'Init', 'Test', 'Seeds', 'Within', 'Cross', 'Comb', 'Comb − within [CI]', 'p Holm', 'Comb − cross [CI]', 'p Holm'], size=6.5)
    doc.add_heading('3. Balanced accuracy and AUC', 1)
    tbl(doc, A, ['variant', 'task', 'model', 'init', 'test_machine', 'within_bacc', 'cross_bacc', 'comb_bacc', 'within_auc', 'cross_auc', 'comb_auc'],
        ['Version', 'Task', 'Backbone', 'Init', 'Test', 'Within BAcc', 'Cross BAcc', 'Comb BAcc', 'Within AUC', 'Cross AUC', 'Comb AUC'], size=6.5)
    doc.save(f'{OUT}/analysis/Phase3_Report.docx')
    lines = [f'Phase 3 - {len(DATA) // 2}/{len(GROUPS)} groups complete, bootstrap {N_BOOT}']
    for _, r in PO.iterrows():
        lines.append(f"{r['version']} | {r['init']} | {r['test_machine']} | within {r['within_f1']:.3f} cross {r['cross_f1']:.3f} comb {r['comb_f1']:.3f} | "
                     f"comb-within {r['comb_minus_within']:+.3f} [{r['comb_minus_within_lo']:+.3f}, {r['comb_minus_within_hi']:+.3f}] p {r['comb_minus_within_p']:.4f} | "
                     f"comb-cross {r['comb_minus_cross']:+.3f} [{r['comb_minus_cross_lo']:+.3f}, {r['comb_minus_cross_hi']:+.3f}] p {r['comb_minus_cross_p']:.4f}")
    open(f'{OUT}/analysis/summary.txt', 'w').write('\n'.join(lines)); print('\n'.join(lines))
    print('Saved:', f'{OUT}/analysis/Phase3_Report.docx')